In [0]:
from pyspark.sql.functions import *

In [0]:


container = "gold-layer"
storage_account = "etlprojectspotify"
mount_name = "gold"

result=dbutils.notebook.run("/Workspace/Users/suman.kr.ghorai@gmail.com/Spotify-ETL-Databricks/utils/mount_utils", 60, {
    "container": container,
    "storage_account": storage_account,
    "mount_name": mount_name
})
print(result)

In [0]:
silver_df = spark.read.format("delta").load("/mnt/silver/spotify_silver/")
display(silver_df)

In [0]:
silver_df.createOrReplaceTempView("silver_spotify")

In [0]:
dim_track = silver_df.select(
    "spotify_id", "name", "duration_ms", "danceability", "energy", "key",
    "key_name", "mode", "mode_name", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo", "time_signature"
).dropDuplicates(["spotify_id"])

dim_track.write.mode("overwrite").format("delta").save("/mnt/gold/dim_track")


In [0]:
dim_artist = silver_df.select(
    "spotify_id", "artists_array", "artist_count"
).dropDuplicates(["spotify_id"])

dim_artist.write.mode("overwrite").format("delta").save("/mnt/gold/dim_artist")


In [0]:
dim_album = silver_df.select(
    "spotify_id", "album_name", "album_release_date",
    "album_release_year", "album_release_month", "album_release_day"
).dropDuplicates(["spotify_id"])

dim_album.write.mode("overwrite").format("delta").save("/mnt/gold/dim_album")


In [0]:
dim_date = silver_df.select(
    "snapshot_date", "snapshot_year", "snapshot_month",
    "snapshot_day", "snapshot_weekday"
).dropDuplicates(["snapshot_date"])

dim_date.write.mode("overwrite").format("delta").save("/mnt/gold/dim_date")


In [0]:
fact_track_performance = silver_df.select(
    "silver_id", "spotify_id", "snapshot_date", "country", "daily_rank",
    "daily_movement", "weekly_movement", "popularity", "is_explicit",
    "duration_sec", "track_age_days", "energy_category", 
    "valence_category", "tempo_category"
)

fact_track_performance.write.mode("overwrite").format("delta").save("/mnt/gold/fact_track_performance")


In [0]:
top_10_tracks = spark.sql("""
SELECT
    snapshot_date, country, spotify_id, name, daily_rank
FROM (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY snapshot_date, country ORDER BY daily_rank ASC) as rank_pos
    FROM silver_spotify
)
WHERE rank_pos <= 10
""")

top_10_tracks.write.mode("overwrite").format("delta").save("/mnt/gold/kpi_top_10_tracks")


In [0]:
avg_popularity_by_artist = silver_df.selectExpr("explode(artists_array) as artist", "popularity") \
    .groupBy("artist") \
    .agg(avg("popularity").alias("avg_popularity")) \
    .orderBy("avg_popularity", ascending=False)

avg_popularity_by_artist.write.mode("overwrite").format("delta").save("/mnt/gold/kpi_avg_popularity_by_artist")


In [0]:
explicit_counts = silver_df.groupBy("country", "is_explicit") \
    .count()

explicit_counts.write.mode("overwrite").format("delta").save("/mnt/gold/kpi_explicit_track_counts")


In [0]:
energy_distribution = silver_df.groupBy("energy_category").count()

energy_distribution.write.mode("overwrite").format("delta").save("/mnt/gold/kpi_energy_distribution")


In [0]:
dbutils.notebook.exit(f"Success:Star Schema,KPi,Aggregate Tables are created and stored in gold layer")